In [9]:
import pandas as pd
import numpy as np

# 1) 读取
df = pd.read_csv("/Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/controlpoints_out/controlpoints_utm_out/merged_Rule8_full.csv")

# 2) 选择平面坐标作为聚类输入（优先 easting/northing；若没有就用 lon/lat 但效果略次）
if {"easting","northing"}.issubset(df.columns):
    X = df[["easting","northing"]].to_numpy()
else:
    X = df[["lon","lat"]].to_numpy()  # 如果没有UTM坐标

# 3) KMeans 聚类，得到恰好 400 个代表点
from sklearn.cluster import KMeans
k = 400
km = KMeans(n_clusters=k, n_init=10, random_state=42)
labels = km.fit_predict(X)
centers = km.cluster_centers_

# 4) 每个簇里选距离质心最近的那个原始点
#    先计算每点到自己簇质心的距离
d = np.linalg.norm(X - centers[labels], axis=1)
df["_label"] = labels
df["_d"] = d
#    对每个簇取距离最小的行
idx = df.sort_values("_d").groupby("_label", as_index=False).head(1).index
sampled = df.loc[idx].copy()

# 5) 生成唯一ID并保存
sampled = sampled.reset_index(drop=True)
sampled.insert(0, "ID", [f"pt{i+1}" for i in range(len(sampled))])
sampled.to_csv("/Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/controlpoints_out/controlpoints_utm_out/merged_Rule8_sampled_400.csv", index=False)
print("选到点数：", len(sampled))

选到点数： 400
